# RTStream V2 Cookbook

Use VideoDB to connect a live RTSP feed, understand it continuously, turn the understanding into a searchable index, and react to events in real time.

The required path is:

**connect to VideoDB → connect the stream → create an understanding → read records → create an index → search → stop resources**

Alerts, transcription, pause/resume, and recording export are clearly marked as optional. Run the required sections from top to bottom. Live resources consume compute while they are running, so always run **Stop and clean up** before leaving the notebook.

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/guides/indexing-v2/rtstream/quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Before you begin

You need:

- a [VideoDB API key](https://console.videodb.io/), supplied through `VIDEO_DB_API_KEY` or entered securely when prompted;
- an RTSP source that VideoDB can reach, or the public sample used below;
- about two to three minutes for the first understanding and index records to appear.

This cookbook uses one VLM analyzer named `scene`. Keeping that output stored is required because the index reads from it.

## 1. Install the RTStream V2 SDK

Run this once per notebook session. If the notebook asks you to restart the kernel after installation, restart it and continue with the next section.

In [ ]:
!pip install -q --force-reinstall --no-cache-dir "git+https://github.com/video-db/videodb-python.git@feat/add-indexing-v2" python-dotenv websockets

## 2. Connect to VideoDB

This is the first Python step. It connects to your default collection immediately—there are no setup helpers to understand first. This public cookbook always uses VideoDB production and intentionally has no dev/staging selector.

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()

api_key = os.getenv("VIDEO_DB_API_KEY") or getpass("Enter your VideoDB API key: ")
if not api_key:
    raise ValueError("A VideoDB API key is required.")

conn = connect(api_key=api_key, base_url="https://api.videodb.io")
collection = conn.get_collection()

print("Connected to VideoDB")
print("Collection ID:", collection.id)

## 3. Configure and connect the RTStream

`connect_rtstream()` creates the live parent resource. VideoDB pulls media from the RTSP URL; the notebook does not upload a file. The returned ID starts with `rts-`.

The defaults form one consistent example: the crib feed, a crib-focused prompt, search query, and alert condition. Change `RTSP_URL` to use your own reachable RTSP source.

- `STORE_RECORDING` retains the live media so it can be exported after the stream stops. Set it to `False` when you do not need the recording.
- `INCLUDE_AUDIO` requests the source's audio track. Enable it only when the source has audio and you plan to use transcription.
- `WINDOW` is the duration of each understanding segment. `"10s"` means the VLM analyzes one 10-second portion of the stream at a time.
- `FRAME_COUNT` is the number of frames sampled evenly across each window and sent to the VLM. With a 10-second window and 5 frames, samples are spaced about 2.5 seconds apart.
- `SCENE_PROMPT` tells the VLM what to describe in every window.

`WINDOW` and `FRAME_COUNT` are passed to `rtstream.understand()` in section 5. Shorter windows produce updates more frequently; more frames provide more visual context but require more processing. The 10-second/5-frame defaults are a practical starting point.

In [ ]:
import time
from datetime import datetime, timezone

RTSP_URL = "rtsp://samples.rts.videodb.io:8554/crib"
STREAM_NAME = "rtstream-v2-cookbook"

STORE_RECORDING = True   # Retain media so the stopped stream can be exported.
INCLUDE_AUDIO = False    # Enable only when the source has audio.

WINDOW = "10s"          # Analyze the stream in 10-second segments.
FRAME_COUNT = 5          # Sample 5 frames evenly across each segment.
SCENE_PROMPT = "Describe the scene clearly. Mention whether a baby or crib is visible and what is happening."

media_types = ["video", "audio"] if INCLUDE_AUDIO else ["video"]
run_started_at = time.time()

rtstream = collection.connect_rtstream(
    url=RTSP_URL,
    name=f"{STREAM_NAME}-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}",
    media_types=media_types,
    store=STORE_RECORDING,
)

print("RTStream ID:", rtstream.id)
print("Status:", rtstream.status)
print("Source:", RTSP_URL)
print("Media types:", media_types)
print("Understanding window:", WINDOW)
print("Frames per window:", FRAME_COUNT)

### The three V2 resources

| Resource | ID prefix | What it does | How to stop or resume it |
|---|---|---|---|
| RTStream | `rts-` | Pulls the live source and optionally retains the recording | `rtstream.stop()` / `rtstream.start()` |
| Understanding | `und-` | Runs analyzers on each live time window | `understanding.stop()` / `understanding.start()` |
| Index | `idx-` | Stores one understanding output for records and search | `index.stop()` / `index.start()` |

They are separate resources. Stopping an index does not stop its understanding or parent stream.

## 4. Open the realtime channel (recommended)

The WebSocket delivers live understanding, alert, and transcript messages. It is recommended for this cookbook but is not required for durable records or search.

A short background listener is necessary in Jupyter so the socket can keep receiving messages while later cells run. The helper below is introduced here because this is the first step that needs it.

In [ ]:
import asyncio
import threading

realtime_messages = []
realtime_ready = threading.Event()
realtime_state = {"error": None}


def run_realtime_listener():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    realtime_state["loop"] = loop

    async def listen():
        socket = conn.connect_websocket(collection.id)
        realtime_state["socket"] = socket
        await socket.connect()
        realtime_state["connection_id"] = socket.connection_id
        realtime_ready.set()
        async for message in socket.receive():
            realtime_messages.append(message)

    try:
        loop.run_until_complete(listen())
    except Exception as exc:
        realtime_state["error"] = f"{type(exc).__name__}: {exc}"
        realtime_ready.set()
    finally:
        loop.close()


realtime_thread = threading.Thread(target=run_realtime_listener, daemon=True)
realtime_thread.start()
realtime_ready.wait(timeout=25)

ws_connection_id = realtime_state.get("connection_id")
if ws_connection_id:
    print("Realtime connection ID:", ws_connection_id)
else:
    print("Realtime connection unavailable:", realtime_state.get("error") or "connection timed out")
    print("Continue normally; only live messages will be unavailable.")

## 5. Create a continuous understanding

The time segmentation below creates one analysis window every 10 seconds. The VLM samples five frames from each window and writes the response to the named output `scene`.

`store=True` is important in this workflow: it makes understanding records durable and allows an index to consume the output.

In [ ]:
understanding = rtstream.understand(
    segmentation={"type": "time", "window": WINDOW},
    analyzers=[
        {
            "type": "vlm",
            "name": "scene",
            "sampling": {"frame_count": FRAME_COUNT},
            "config": {"prompt": SCENE_PROMPT},
        }
    ],
    store=True,
    ws_connection_id=ws_connection_id,
)

print("Understanding ID:", understanding.id)
print("Status:", understanding.status)
print("Available outputs:", list(understanding.outputs))
print("Scene output descriptor:", understanding.outputs["scene"])

### Retrieve existing understanding resources

Use `get_understanding(id)` when you know the ID and `list_understanding()` when you need every V2 understanding attached to this RTStream. The SDK method is singular: `list_understanding()`.

In [ ]:
same_understanding = rtstream.get_understanding(understanding.id)
all_understandings = rtstream.list_understanding()

print("Fetched ID:", same_understanding.id)
print("Understanding IDs on this stream:", [item.id for item in all_understandings])

## 6. Observe live understanding messages

Understanding messages arrive on the `visual_index` channel. The first result may take several windows while the live worker and VLM warm up.

This cell waits for up to two minutes. If it times out, continue to the durable-record section and rerun this cell later.

In [ ]:
def messages_on(channel):
    return [
        message
        for message in realtime_messages
        if isinstance(message, dict) and message.get("channel") == channel
    ]


if not ws_connection_id:
    print("Skipped because the realtime channel is unavailable.")
else:
    deadline = time.monotonic() + 120
    while time.monotonic() < deadline and not messages_on("visual_index"):
        print("Waiting for the first live understanding message...")
        time.sleep(5)

    visual_messages = messages_on("visual_index")
    print()
    print("Understanding messages received:", len(visual_messages))
    if visual_messages:
        print("Latest message:", visual_messages[-1])
    else:
        print("No message yet. The pipeline may still be warming up.")

## 7. Read durable understanding records

`understanding.get_records()` reads stored analyzer output for a Unix-timestamp range. Unlike WebSocket messages, these records remain available after the realtime connection closes.

The polling helper is defined next to the read that needs it. It tolerates the normal warm-up period and returns as soon as records exist.

In [ ]:
def extract_records(payload):
    if isinstance(payload, list):
        return payload
    if not isinstance(payload, dict):
        return []
    for key in ("records", "scene_index_records", "scenes", "segments", "transcription_records"):
        if isinstance(payload.get(key), list):
            return payload[key]
    return []


def wait_for_records(fetch, label, timeout=180, interval=10):
    deadline = time.monotonic() + timeout
    last_error = None

    while time.monotonic() < deadline:
        try:
            payload = fetch()
            records = extract_records(payload)
            if records:
                print(f"{label}: {len(records)} record(s)")
                return payload, records
            print(f"Waiting for {label}...")
        except Exception as exc:
            last_error = exc
            print(f"Waiting for {label} ({type(exc).__name__})...")
        time.sleep(interval)

    print()
    if last_error:
        print(f"No {label} before timeout. Last error: {last_error}")
    else:
        print(f"No {label} before timeout. The live pipeline may still be warming up.")
    return {}, []


record_window_start = run_started_at - 60
understanding_payload, understanding_records = wait_for_records(
    lambda: understanding.get_records(
        start=record_window_start,
        end=time.time(),
        output="scene",
        page=1,
        page_size=100,
    ),
    label="understanding records",
)

for record in understanding_records[:5]:
    print(record)

## 8. Create an index from the `scene` output

An understanding can expose one or more named outputs. An index consumes one output descriptor—not the entire understanding.

`use_for=["semantic"]` makes these records available to semantic search.

In [ ]:
scene_output = understanding.outputs["scene"]

index = rtstream.index(
    source=scene_output,
    name="rtstream-v2-scenes",
    use_for=["semantic"],
)

print("Index ID:", index.id)
print("Status:", index.status)
print("Source understanding ID:", index.source_understanding_id)
print("Capabilities:", index.use_for)

### Retrieve existing indexes

Use `get_index(id)` for one V2 index and `list_indexes()` for all V2 indexes on this RTStream.

In [ ]:
same_index = rtstream.get_index(index.id)
all_indexes = rtstream.list_indexes()

print("Fetched ID:", same_index.id)
print("Index IDs on this stream:", [item.id for item in all_indexes])

### Read materialized index records

Index records are the searchable form of the understanding output. They can appear shortly after understanding records, so this read polls independently.

In [ ]:
index_payload, index_records = wait_for_records(
    lambda: index.get_records(
        start=record_window_start,
        end=time.time(),
        page=1,
        page_size=100,
    ),
    label="index records",
)

for record in index_records[:5]:
    print(record)

### V2 and legacy indexes are different

For this workflow, use `list_indexes()`. `list_scene_indexes()` belongs to the earlier RTStream scene-index API and does not return V2 `idx-` resources.

In [ ]:
legacy_scene_indexes = rtstream.list_scene_indexes()

print("V2 indexes:", [item.id for item in all_indexes])
print(
    "Legacy scene indexes:",
    [getattr(item, "id", None) or getattr(item, "rtstream_index_id", None) for item in legacy_scene_indexes],
)

## 9. Search the live index

Search after index records exist. Use a query that matches the selected source; the default query matches the crib sample.

Each result is an `RTStreamShot` containing the matching time range, text, and similarity score.

In [ ]:
SEARCH_QUERY = "a baby or crib is visible"

search_result = rtstream.search(
    query=SEARCH_QUERY,
    index_id=index.id,
    result_threshold=5,
)
shots = search_result.get_shots()

print(f"Results for {SEARCH_QUERY!r}: {len(shots)}")
for shot in shots:
    print(f"[{shot.start} -> {shot.end}] score={shot.search_score} | {shot.text}")

### Generate a playable result

`shot.generate_stream()` creates a media URL for that result window and also sets `shot.player_url`. Run this only after search returns at least one shot.

In [ ]:
if not shots:
    print("No search result is available yet. Wait for more index records and rerun the search.")
else:
    first_shot = shots[0]
    search_stream_url = first_shot.generate_stream()
    print("Media URL:", search_stream_url)
    print("Player URL:", first_shot.player_url)

## 10. Optional — create a live alert

An event defines the condition; an alert applies that event to this index. A callback URL is required even when you also pass the WebSocket connection ID.

Set `RTSTREAM_ALERT_CALLBACK_URL` in the environment or replace the empty value below with your webhook URL. If no URL is supplied, the cell safely skips alert creation.

In [ ]:
EVENT_PROMPT = "A baby crib is visible in the frame"
EVENT_LABEL = "crib-visible"
CALLBACK_URL = os.getenv("RTSTREAM_ALERT_CALLBACK_URL", "").strip()

alert_id = None
event_id = None

if not CALLBACK_URL:
    print("Skipped. Set RTSTREAM_ALERT_CALLBACK_URL and rerun this cell.")
else:
    event_id = conn.create_event(event_prompt=EVENT_PROMPT, label=EVENT_LABEL)
    alert_id = index.create_alert(
        event_id=event_id,
        callback_url=CALLBACK_URL,
        ws_connection_id=ws_connection_id,
    )
    print("Event ID:", event_id)
    print("Alert ID:", alert_id)
    print("Alerts on this index:", index.list_alerts())

### Enable or disable an alert

A new alert is enabled when it is created. Use these methods to control delivery without deleting the index. This demonstration is opt-in so the active alert is not interrupted accidentally.

In [ ]:
RUN_ALERT_TOGGLE_DEMO = False

if not alert_id:
    print("Skipped because no alert was created.")
elif not RUN_ALERT_TOGGLE_DEMO:
    print("Alert remains enabled. Set RUN_ALERT_TOGGLE_DEMO=True to test disable and enable.")
else:
    index.disable_alert(alert_id)
    print("Alert disabled")
    index.enable_alert(alert_id)
    print("Alert enabled again")
    print(index.list_alerts())

### Inspect a fired alert and generate its clip

Alert WebSocket messages use the `alert` channel. A fired alert may already include `player_url` and `stream_url`.

`rtstream.generate_stream(start, end)` requests a fresh clip for the same timestamps. The method returns the player URL and sets both `rtstream.player_url` and `rtstream.stream_url`.

In [ ]:
alert_messages = messages_on("alert") if ws_connection_id else []

if not alert_messages:
    print("No alert message is available. Leave the stream running and rerun this cell after the condition occurs.")
else:
    latest_alert = alert_messages[-1]
    alert_data = latest_alert.get("data") or latest_alert

    print("Label:", alert_data.get("label"))
    print("Window:", alert_data.get("start"), "->", alert_data.get("end"))
    print("Delivered media URL:", alert_data.get("stream_url"))
    print("Delivered player URL:", alert_data.get("player_url"))

    if alert_data.get("start") is None or alert_data.get("end") is None:
        print("This alert has no complete time window, so a new clip cannot be generated.")
    else:
        generated_player_url = rtstream.generate_stream(
            start=int(alert_data["start"]),
            end=int(alert_data["end"]),
        )
        print("Generated media URL:", rtstream.stream_url)
        print("Generated player URL:", generated_player_url)

## 11. Optional — live transcription

Run this section only if the RTSP source has audio and you set `INCLUDE_AUDIO=True` **before** creating the RTStream.

`start_transcript()` begins live transcription. WebSocket delivery is immediate; `get_transcript()` reads durable finalized segments. The cleanup section stops transcription automatically when it was started here.

In [ ]:
transcription_started = False
transcript_records = []

if not INCLUDE_AUDIO:
    print("Skipped. Set INCLUDE_AUDIO=True, reconnect the RTStream, then rerun from section 3.")
else:
    transcript_status = rtstream.start_transcript(
        ws_connection_id=ws_connection_id,
        engine="assemblyai",
    )
    transcription_started = True
    print("Transcription started:", transcript_status)

    transcript_payload, transcript_records = wait_for_records(
        lambda: rtstream.get_transcript(
            page=1,
            page_size=1000,
            start=record_window_start,
            end=time.time(),
            engine="assemblyai",
        ),
        label="transcript records",
        timeout=120,
    )

    for record in transcript_records[:5]:
        print(record)

## 12. Optional — pause and resume processing

An understanding and its index have independent lifecycles. Stopping either job pauses new processing but keeps existing records. This demonstration is opt-in because pausing the live pipeline creates a gap in new results.

In [ ]:
RUN_PAUSE_RESUME_DEMO = False

if not RUN_PAUSE_RESUME_DEMO:
    print("Set RUN_PAUSE_RESUME_DEMO=True to run this lifecycle demonstration.")
else:
    index.stop()
    understanding.stop()
    print("Paused index and understanding")

    understanding.start()
    index.start()
    print("Resumed understanding and index")
    print("Understanding status:", rtstream.get_understanding(understanding.id).status)
    print("Index status:", rtstream.get_index(index.id).status)

## 13. Stop and clean up — always run this

Run this cell even if an earlier step failed. It attempts each cleanup action independently in this order:

1. disable the alert;
2. stop transcription;
3. stop the index and understanding;
4. stop the RTStream;
5. close the WebSocket.

Stopping compute does not delete stored understanding or index records.

In [ ]:
cleanup_results = {}

if globals().get("alert_id"):
    try:
        index.disable_alert(alert_id)
        cleanup_results["alert"] = "disabled"
    except Exception as exc:
        cleanup_results["alert"] = f"disable failed: {exc}"

if globals().get("transcription_started"):
    try:
        rtstream.stop_transcript(engine="assemblyai")
        cleanup_results["transcription"] = "stopped"
    except Exception as exc:
        cleanup_results["transcription"] = f"stop failed: {exc}"

for label, resource in (
    ("index", globals().get("index")),
    ("understanding", globals().get("understanding")),
    ("rtstream", globals().get("rtstream")),
):
    if resource is None:
        continue
    try:
        resource.stop()
        cleanup_results[label] = "stopped"
    except Exception as exc:
        cleanup_results[label] = f"stop failed: {exc}"

loop = realtime_state.get("loop") if "realtime_state" in globals() else None
socket = realtime_state.get("socket") if "realtime_state" in globals() else None
if loop and socket and loop.is_running():
    try:
        asyncio.run_coroutine_threadsafe(socket.close(), loop).result(timeout=10)
        cleanup_results["websocket"] = "closed"
    except Exception as exc:
        cleanup_results["websocket"] = f"close failed: {exc}"

print(cleanup_results)

## 14. Verify stored data after stopping

These reads show the durability boundary: stopping live compute does not erase stored resources and records. Each read is isolated so one unavailable optional surface does not hide the others.

In [ ]:
def summarize(resource):
    return {
        "id": getattr(resource, "id", None),
        "name": getattr(resource, "name", None),
        "status": getattr(resource, "status", None),
    }


def safe_read(label, fetch):
    try:
        value = fetch()
        print(f"{label}: OK")
        return value
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        print(f"{label}: {error}")
        return {"error": error}


post_stop_end = time.time()
post_stop_reads = {
    "stream": safe_read("stream", lambda: summarize(collection.get_rtstream(rtstream.id))),
    "understandings": safe_read(
        "understandings",
        lambda: [summarize(item) for item in rtstream.list_understanding()],
    ),
    "understanding": safe_read(
        "understanding",
        lambda: summarize(rtstream.get_understanding(understanding.id)),
    ),
    "understanding_records": safe_read(
        "understanding records",
        lambda: understanding.get_records(
            start=record_window_start,
            end=post_stop_end,
            output="scene",
            page_size=100,
        ),
    ),
    "indexes": safe_read("indexes", lambda: [summarize(item) for item in rtstream.list_indexes()]),
    "index": safe_read("index", lambda: summarize(rtstream.get_index(index.id))),
    "index_records": safe_read(
        "index records",
        lambda: index.get_records(start=record_window_start, end=post_stop_end, page_size=100),
    ),
    "alerts": safe_read("alerts", lambda: index.list_alerts()),
    "transcript": safe_read(
        "transcript",
        lambda: rtstream.get_transcript(
            start=record_window_start,
            end=post_stop_end,
            page_size=100,
        ),
    ),
}

print("Post-stop reads complete. Use post_stop_reads to inspect full payloads.")

## 15. Optional — export the retained recording

This works only when `STORE_RECORDING=True`, and only after the RTStream is stopped. Recording finalization is asynchronous, so the cell retries for up to about 30 seconds.

Export is idempotent: rerunning it returns the same VideoDB asset rather than creating a duplicate.

In [ ]:
if not STORE_RECORDING:
    print("Skipped because this RTStream was created with STORE_RECORDING=False.")
else:
    export_result = None
    last_export_error = None

    for attempt in range(1, 7):
        try:
            export_result = rtstream.export(name="RTStream V2 Cookbook Recording")
            break
        except Exception as exc:
            last_export_error = exc
            print(f"Export attempt {attempt}/6 is not ready: {exc}")
            time.sleep(5)

    if export_result is None:
        print("The recording did not finalize during the retry window:", last_export_error)
    else:
        print("Video ID:", export_result.video_id)
        print("Duration:", export_result.duration)
        print("Media URL:", export_result.stream_url)
        print("Player URL:", export_result.player_url)

## Quick reference

| Goal | Public SDK call |
|---|---|
| Connect a live source | `collection.connect_rtstream(...)` |
| Open live updates | `conn.connect_websocket(collection.id)` |
| Start understanding | `rtstream.understand(...)` |
| Retrieve understandings | `rtstream.get_understanding(id)` / `rtstream.list_understanding()` |
| Read understanding output | `understanding.get_records(start, end, output="scene")` |
| Create a V2 index | `rtstream.index(source=understanding.outputs["scene"])` |
| Retrieve V2 indexes | `rtstream.get_index(id)` / `rtstream.list_indexes()` |
| Read indexed records | `index.get_records(start, end)` |
| Search | `rtstream.search(query, index_id=index.id)` |
| Create and control an alert | `index.create_alert(...)`, `disable_alert(id)`, `enable_alert(id)` |
| Start or stop transcription | `rtstream.start_transcript(...)` / `rtstream.stop_transcript(...)` |
| Read transcript segments | `rtstream.get_transcript(...)` |
| Generate a time-range player | `rtstream.generate_stream(start, end)` |
| Export a retained recording | `rtstream.export()` after `rtstream.stop()` |

## What you built

You connected one live source and composed three independent V2 resources:

**RTStream (`rts-`) → Understanding output (`und-`) → searchable Index (`idx-`)**

From that index, you read durable records, searched live scenes, and optionally attached an alert. You also saw where WebSocket messages, transcription, clip generation, recording export, and lifecycle controls fit without mixing them into the required path.